# Clasificación de anemia infantil con Árboles de Decisión

**Proyecto de Minería de Datos — Modelo de Clasificación con Árboles de Decisión**

Este cuaderno entrena y evalúa un árbol de decisión que predice si un niño presenta **algún grado de anemia** a partir de variables demográficas, sociales y de cuidado infantil. Sigue las cinco secciones pedidas en la entrega:

1. Definición del problema y variable objetivo
2. Preparación de los datos
3. Entrenamiento del modelo (árbol base + ajuste de hiperparámetros)
4. Evaluación del modelo
5. Visualización y comunicación de hallazgos

> **Cómo leer el cuaderno:** cada sección tiene una explicación antes del código y una interpretación después. Las celdas de código imprimen sus propias conclusiones con los números reales, de modo que el texto siempre coincide con los resultados.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             precision_recall_curve)
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     StratifiedKFold, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

SEED = 42
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

---
# 1. Definición del problema y variable objetivo

### Problema de investigación
La anemia infantil afecta el crecimiento y el desarrollo cognitivo de los niños. Detectarla a tiempo permite intervenir (suplementación con hierro, educación nutricional, seguimiento). La pregunta que guía este trabajo es:

> **¿Se puede identificar, a partir de las características del niño, de su familia y de su entorno, qué niños tienen mayor probabilidad de presentar anemia, y cuáles de esas características pesan más en la decisión?**

### Variable objetivo (dependiente): `Con_Anemia`
- Se construye a partir de `Grado_Anemia`: **1** = algún grado de anemia (leve, moderada o severa) y **0** = sin anemia.
- **Por qué binaria:** el objetivo de un tamizaje es decidir *quién necesita atención*, no clasificar el grado exacto. Agrupar los grados también evita clases con muy pocos casos, que un árbol aprendería mal.

### Variables independientes (explicativas) y su justificación

| Grupo | Variables | Por qué podrían influir |
|---|---|---|
| Territorio | `Departamento`, `Área` | La prevalencia de anemia varía por región, altitud y acceso a servicios (urbano/rural). |
| Perfil del niño | `Sexo`, `Grupo_Etario` | Las necesidades de hierro y el riesgo cambian con la edad. |
| Cuidado y nutrición | `Consume_Suplemento_Hierro`, `Lactancia_Materna_Exclusiva` | Son factores directamente ligados a las reservas de hierro. |
| Contexto familiar | `Nivel_Educativo_Madre` | Se asocia con prácticas de alimentación y acceso a información de salud. |
| Variables numéricas | Todas las demás columnas numéricas disponibles | Se incluyen automáticamente (ver sección 2) tras descartar las que filtran la respuesta. |

### Variables que se EXCLUYEN a propósito (fuga de información, *data leakage*)
- `Hemoglobina_g_dL` y `Grado_Anemia`: la anemia **se define** por el nivel de hemoglobina. Incluirlas haría que el modelo "descubra" una regla trivial y no aprendería nada útil.
- `Deteccion_Tardia_Anemia`: solo existe *después* de haber detectado la anemia, así que no está disponible cuando se quiere predecir.
- `ID_Registro` y `Fecha_Tamizaje`: identificadores y fechas sin valor explicativo; podrían inducir al árbol a memorizar registros.

In [ ]:
RUTA = "anemia_infantil_datos.xlsx"   # ajusta la ruta si el archivo está en otra carpeta
df = pd.read_excel(RUTA, header=3)    # header=3: los nombres de columna están en la fila 4 del Excel

print("Dimensiones originales:", df.shape)
display(df.head())
print()
df.info()

**Interpretación.** Esta primera mirada confirma el tamaño del conjunto de datos, el tipo de cada columna y si hay columnas con valores faltantes. Con esto se decide qué columnas son categóricas, cuáles numéricas y cuáles hay que excluir.

---
# 2. Preparación de los datos

Pasos, en este orden:
1. **Limpieza:** duplicados y valores nulos (se informan por columna).
2. **Variable objetivo:** se crea `Con_Anemia` y se revisa el balance de clases.
3. **Exploración breve:** tasa de anemia por categoría, para tener una intuición antes de modelar.
4. **División entrenamiento/prueba (80/20, estratificada).** Se hace **antes** de imputar y codificar, para que ninguna estadística del conjunto de prueba influya en el entrenamiento.
5. **Codificación y relleno de nulos dentro de un `Pipeline`:** los nulos se rellenan (mediana en numéricas, moda en categóricas) y las categóricas se convierten en columnas 0/1 (*one-hot*). Ambos pasos se **aprenden solo con el entrenamiento** y se aplican igual al conjunto de prueba.

In [ ]:
# 2.1 Limpieza ---------------------------------------------------------------
n0 = len(df)
if "ID_Registro" in df.columns:
    df = df.drop_duplicates(subset="ID_Registro")
print(f"Registros duplicados eliminados: {n0 - len(df)}")

nulos = df.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
if nulos.empty:
    print("No hay valores nulos en ninguna columna.")
else:
    print("Valores nulos por columna (se imputarán dentro del Pipeline):")
    display(pd.DataFrame({"nulos": nulos, "%": (nulos / len(df) * 100).round(2)}))

# 2.2 Variable objetivo ------------------------------------------------------
df = df[df["Grado_Anemia"].notna()].copy()           # sin grado no hay etiqueta
df["Con_Anemia"] = (df["Grado_Anemia"] != "Sin anemia").astype(int)

balance = df["Con_Anemia"].value_counts().rename({0: "Sin anemia", 1: "Con anemia"})
print("\nBalance de clases:")
display(pd.DataFrame({"casos": balance, "%": (balance / len(df) * 100).round(1)}))

In [ ]:
# 2.3 Exploración rápida: tasa de anemia por categoría -----------------------
cat_cols = ["Departamento", "Área", "Sexo", "Consume_Suplemento_Hierro",
            "Lactancia_Materna_Exclusiva", "Nivel_Educativo_Madre", "Grupo_Etario"]
cat_cols = [c for c in cat_cols if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), cat_cols):
    tasa = df.groupby(col)["Con_Anemia"].mean().sort_values()
    ax.barh(tasa.index.astype(str), tasa.values * 100, color="#4C78A8")
    ax.axvline(df["Con_Anemia"].mean() * 100, color="crimson", ls="--", lw=1)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("% con anemia")
for ax in axes.ravel()[len(cat_cols):]:
    ax.axis("off")
fig.suptitle("Tasa de anemia por categoría (línea roja = promedio general)", y=1.01)
plt.tight_layout()
plt.show()

**Interpretación.** Si todas las barras de un gráfico están cerca de la línea roja, esa variable discrimina poco: las diferencias entre categorías son pequeñas y un árbol tendrá dificultad para separar a los niños con y sin anemia usándola. Las variables con barras claramente más largas o más cortas que el promedio son las candidatas a aparecer en las primeras divisiones del árbol.

In [ ]:
# 2.4 Selección de variables y división train/test ---------------------------
excluir = ["ID_Registro", "Fecha_Tamizaje", "Hemoglobina_g_dL", "Grado_Anemia",
           "Con_Anemia", "Deteccion_Tardia_Anemia"]

num_cols = [c for c in df.select_dtypes(include="number").columns
            if c not in excluir and c not in cat_cols]
features = cat_cols + num_cols

# Cualquier columna restante que no sea ni categórica ni numérica (para no perderla sin darnos cuenta)
sin_usar = [c for c in df.columns if c not in features and c not in excluir]
print("Variables categóricas :", cat_cols)
print("Variables numéricas   :", num_cols)
if sin_usar:
    print("Columnas no utilizadas (revisar si deberían entrar):", sin_usar)

X = df[features].copy()
X[cat_cols] = X[cat_cols].astype("object")
y = df["Con_Anemia"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)

print(f"\nEntrenamiento: {X_train.shape}  |  Prueba: {X_test.shape}")
print(f"% con anemia -> train: {y_train.mean():.1%}  |  test: {y_test.mean():.1%}")

In [ ]:
# 2.5 Preprocesamiento (imputación + codificación) ---------------------------
def crear_preprocesador(escalar=False):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if escalar:                      # el árbol no lo necesita; la regresión logística sí
        num_steps.append(("esc", StandardScaler()))
    return ColumnTransformer(
        [("num", Pipeline(num_steps), num_cols),
         ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                           ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
          cat_cols)],
        verbose_feature_names_out=False)

prep_demo = crear_preprocesador().fit(X_train)
nombres_features = prep_demo.get_feature_names_out()
print(f"{len(features)} variables originales -> {len(nombres_features)} columnas tras la codificación")

**Interpretación.** El conjunto queda dividido en 80 % para entrenar y 20 % para evaluar, conservando la misma proporción de casos con anemia en ambos (`stratify=y`). La codificación *one-hot* mantiene **todas** las categorías (no se elimina una de referencia como en la regresión), porque en un árbol cada columna 0/1 es una pregunta del tipo "¿vive en el departamento X?" y así el resultado es más fácil de leer.

---
# 3. Entrenamiento del modelo de Árbol de Decisión

### 3.1 Línea base: árbol con parámetros por defecto
Un árbol sin restricciones sigue dividiendo hasta que cada hoja es pura, es decir, **memoriza** el entrenamiento. Se entrena primero para tener un punto de comparación y mostrar el problema del sobreajuste (*overfitting*). También se incluye un clasificador "tonto" que siempre predice la clase mayoritaria: cualquier modelo útil debe superarlo.

In [ ]:
def pipeline_arbol(**params):
    return Pipeline([("prep", crear_preprocesador()),
                     ("arbol", DecisionTreeClassifier(random_state=SEED, **params))])

# Referencia: predecir siempre la clase mayoritaria
dummy = Pipeline([("prep", crear_preprocesador()),
                  ("m", DummyClassifier(strategy="most_frequent"))]).fit(X_train, y_train)

# Árbol por defecto
arbol_base = pipeline_arbol().fit(X_train, y_train)

filas = []
for nombre, m in [("Clase mayoritaria", dummy), ("Árbol por defecto", arbol_base)]:
    filas.append({"Modelo": nombre,
                  "Accuracy train": accuracy_score(y_train, m.predict(X_train)),
                  "Accuracy test":  accuracy_score(y_test,  m.predict(X_test)),
                  "AUC test": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])})
display(pd.DataFrame(filas).round(3))

arbol = arbol_base.named_steps["arbol"]
print(f"Árbol por defecto: profundidad = {arbol.get_depth()}, hojas = {arbol.get_n_leaves()}")

**Interpretación.** Si el árbol por defecto obtiene una exactitud casi perfecta en entrenamiento pero mucho menor en prueba, está sobreajustado: aprendió el "ruido" de los datos de entrenamiento y no generaliza. Por eso es necesario limitar su complejidad con hiperparámetros.

### 3.2 Validación cruzada y ajuste de hiperparámetros
Se prueban combinaciones de los hiperparámetros que controlan la complejidad del árbol, usando **validación cruzada estratificada de 5 particiones** sobre el conjunto de entrenamiento (el de prueba no se toca):

| Hiperparámetro | Qué controla |
|---|---|
| `criterion` | Medida de impureza para elegir la división: `gini` o `entropy`. |
| `max_depth` | Profundidad máxima. Menos profundidad = árbol más simple. |
| `min_samples_leaf` | Mínimo de niños por hoja. Evita hojas con muy pocos casos. |
| `min_samples_split` | Mínimo de niños que debe tener un nodo para poder dividirse. |
| `class_weight` | `balanced` da más peso a la clase minoritaria, lo que ayuda al recall. |

**Métrica de selección:** `roc_auc`. A diferencia de la exactitud, no depende del umbral de decisión ni se infla cuando las clases están desbalanceadas. El recall y el F1 se registran también para poder consultarlos.

In [ ]:
param_grid = {
    "arbol__criterion": ["gini", "entropy"],
    "arbol__max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "arbol__min_samples_leaf": [1, 5, 10, 20, 50],
    "arbol__min_samples_split": [2, 10, 30],
    "arbol__class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid = GridSearchCV(
    pipeline_arbol(), param_grid, cv=cv, n_jobs=-1,
    scoring={"roc_auc": "roc_auc", "recall": "recall", "f1": "f1", "accuracy": "accuracy"},
    refit="roc_auc", return_train_score=True)
grid.fit(X_train, y_train)

print("Mejores hiperparámetros:")
for k, v in grid.best_params_.items():
    print(f"   {k.replace('arbol__', ''):<18} = {v}")
print(f"\nAUC medio en validación cruzada: {grid.best_score_:.3f}")

resultados = pd.DataFrame(grid.cv_results_)
cols = ["params", "mean_test_roc_auc", "mean_test_recall", "mean_test_f1", "mean_test_accuracy"]
print("\nTop 5 combinaciones:")
display(resultados.sort_values("mean_test_roc_auc", ascending=False)[cols].head(5).round(3))

In [ ]:
# ¿Cómo cambia el desempeño con la profundidad? (train vs validación)
res = resultados.copy()
res["profundidad"] = res["param_arbol__max_depth"].apply(lambda d: 15 if d is None else d)
curva = res.groupby("profundidad")[["mean_train_roc_auc", "mean_test_roc_auc"]].max()

plt.figure(figsize=(7, 4.5))
plt.plot(curva.index, curva["mean_train_roc_auc"], "o-", label="Entrenamiento")
plt.plot(curva.index, curva["mean_test_roc_auc"], "o-", label="Validación cruzada")
plt.xticks(curva.index, [("sin límite" if i == 15 else i) for i in curva.index])
plt.xlabel("max_depth")
plt.ylabel("AUC (mejor combinación para cada profundidad)")
plt.title("Sobreajuste vs. profundidad del árbol")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretación.** Cuando la profundidad crece, el AUC de entrenamiento sigue subiendo, pero el de validación se estanca o baja: esa **brecha** es el sobreajuste. La mejor profundidad es la que maximiza el AUC de validación, no la de entrenamiento. El árbol elegido es más simple que el de referencia y, por lo tanto, más fiable con niños nuevos.

In [ ]:
modelo = grid.best_estimator_            # árbol final, ya re-entrenado con todo el train
arbol_final = modelo.named_steps["arbol"]
print(f"Árbol final: profundidad = {arbol_final.get_depth()}, hojas = {arbol_final.get_n_leaves()}")

---
# 4. Evaluación del modelo

### 4.1 Elección del umbral de decisión (sin usar el conjunto de prueba)
El árbol entrega una **probabilidad** de anemia para cada niño; para clasificar hay que fijar un **umbral**. En un tamizaje es más grave **no detectar** a un niño con anemia (falso negativo) que revisar a uno sano (falso positivo), así que conviene buscar un recall alto.

> **Mejora importante respecto a la versión anterior:** el umbral ya no se calcula con el conjunto de prueba (eso lo "contamina" y hace que el resultado parezca mejor de lo que es). Ahora se calcula con predicciones *out-of-fold* del entrenamiento, y el conjunto de prueba se usa **una sola vez** al final.

Se comparan tres reglas: umbral estándar **0.50**, umbral de **Youden** (equilibra sensibilidad y especificidad) y el umbral que **alcanza el recall objetivo** con la mayor precisión posible.

In [ ]:
RECALL_OBJETIVO = 0.82     # meta definida para el proyecto
AUC_OBJETIVO    = 0.85

# Probabilidades out-of-fold sobre el entrenamiento
p_oof = cross_val_predict(modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

# (a) Youden
fpr_o, tpr_o, thr_o = roc_curve(y_train, p_oof)
umbral_youden = thr_o[np.argmax(tpr_o - fpr_o)]

# (b) Mayor umbral que aún cumple el recall objetivo
prec_o, rec_o, thr_pr = precision_recall_curve(y_train, p_oof)
ok = np.where(rec_o[:-1] >= RECALL_OBJETIVO)[0]
umbral_recall = thr_pr[ok[-1]] if len(ok) else 0.0

umbrales = {"Estándar (0.50)": 0.50,
            "Youden": umbral_youden,
            f"Recall ≥ {RECALL_OBJETIVO}": umbral_recall}
print({k: round(float(v), 3) for k, v in umbrales.items()})

### 4.2 Métricas de clasificación en el conjunto de prueba
Definiciones (en el contexto del problema):
- **Accuracy:** proporción de niños clasificados correctamente.
- **Precision:** de los niños que el modelo señala con anemia, ¿cuántos realmente la tienen? (penaliza falsas alarmas).
- **Recall (sensibilidad):** de los niños que realmente tienen anemia, ¿cuántos detecta el modelo? (penaliza casos perdidos).
- **Especificidad:** de los niños sanos, ¿cuántos se identifican como sanos?
- **F1:** media armónica de precision y recall.
- **AUC:** capacidad de ordenar a los niños con anemia por encima de los sanos, independientemente del umbral.

In [ ]:
p_test = modelo.predict_proba(X_test)[:, 1]
auc_arbol = roc_auc_score(y_test, p_test)

def metricas(y_real, p, umbral):
    pred = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_real, pred).ravel()
    return {"Umbral": umbral,
            "Accuracy": accuracy_score(y_real, pred),
            "Precision": precision_score(y_real, pred, zero_division=0),
            "Recall": recall_score(y_real, pred),
            "Especificidad": tn / (tn + fp),
            "F1": f1_score(y_real, pred),
            "Falsos negativos": fn, "Falsos positivos": fp}

tabla = pd.DataFrame({nombre: metricas(y_test, p_test, u) for nombre, u in umbrales.items()}).T
display(tabla.round(3))
print(f"AUC en prueba (no depende del umbral): {auc_arbol:.3f}")
print(f"Meta AUC ≥ {AUC_OBJETIVO}: {'CUMPLIDA' if auc_arbol >= AUC_OBJETIVO else 'NO CUMPLIDA'}")
r_rec = tabla.loc[f'Recall ≥ {RECALL_OBJETIVO}', 'Recall']
print(f"Meta Recall ≥ {RECALL_OBJETIVO} (en prueba, con el umbral elegido en train): "
      f"{r_rec:.3f} -> {'CUMPLIDA' if r_rec >= RECALL_OBJETIVO else 'NO CUMPLIDA'}")

In [ ]:
# Reporte completo con el umbral de Youden (equilibrado) y comparación con el árbol "por defecto"
u_sel = umbrales["Youden"]
print("Reporte de clasificación — árbol ajustado (umbral Youden)\n")
print(classification_report(y_test, (p_test >= u_sel).astype(int),
                            target_names=["Sin anemia", "Con anemia"], digits=3))

p_base = arbol_base.predict_proba(X_test)[:, 1]
print(f"AUC prueba  ->  árbol por defecto: {roc_auc_score(y_test, p_base):.3f}   |   "
      f"árbol ajustado: {auc_arbol:.3f}")

**Interpretación.**
- **Matriz de confusión y falsos negativos:** cada falso negativo es un niño con anemia que el modelo habría dejado pasar. Es la cifra más importante en un tamizaje.
- **Compromiso precision–recall:** al bajar el umbral para alcanzar el recall objetivo, el modelo detecta más casos pero genera más falsas alarmas (la precision cae). No hay un umbral "correcto" universal: depende de cuánto cuesta revisar a un niño sano frente a perder uno con anemia.
- **Metas del proyecto:** si el AUC queda por debajo de 0.85, significa que, con las variables disponibles, el árbol no separa bien a los dos grupos. Esto **no es un error del código**: indica que las variables explicativas (sociales y de cuidado) por sí solas tienen poder predictivo limitado, algo habitual cuando la hemoglobina —la verdadera causa del diagnóstico— se excluye deliberadamente para evitar fuga de información.
- **Árbol por defecto vs. ajustado:** el ajuste de hiperparámetros busca mejorar la generalización; si el AUC en prueba es similar, el principal beneficio es un árbol más simple e interpretable.

### 4.3 Importancia de variables
Se calcula de dos maneras:
1. **Importancia por impureza** (la que da el propio árbol): cuánto reduce la impureza cada columna en sus divisiones. Puede favorecer a variables con muchas categorías o valores.
2. **Importancia por permutación** (en el conjunto de prueba): cuánto cae el AUC si se *desordenan* los valores de una variable. Se mide sobre las variables originales (no sobre cada columna 0/1) y es más fiable para decidir qué variables importan realmente.

In [ ]:
# 1) Importancia por impureza (columnas codificadas)
imp_df = (pd.DataFrame({"Variable": nombres_features,
                        "Importancia": arbol_final.feature_importances_})
          .sort_values("Importancia", ascending=False))

# 2) Importancia por permutación (variables originales)
perm = permutation_importance(modelo, X_test, y_test, scoring="roc_auc",
                              n_repeats=20, random_state=SEED, n_jobs=-1)
perm_df = (pd.DataFrame({"Variable": X_test.columns,
                         "Caída de AUC": perm.importances_mean,
                         "Desv": perm.importances_std})
           .sort_values("Caída de AUC", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
top = imp_df.head(10)
axes[0].barh(top["Variable"], top["Importancia"], color="#4C78A8")
axes[0].invert_yaxis()
axes[0].set_title("Importancia por impureza (top 10)")
axes[0].set_xlabel("Importancia")

axes[1].barh(perm_df["Variable"].head(10), perm_df["Caída de AUC"].head(10),
             xerr=perm_df["Desv"].head(10), color="#F58518")
axes[1].invert_yaxis()
axes[1].set_title("Importancia por permutación (top 10)")
axes[1].set_xlabel("Caída del AUC al desordenar la variable")
plt.tight_layout()
plt.show()

print("Variables con importancia real (la caída de AUC supera 2 desviaciones):")
reales = perm_df[perm_df["Caída de AUC"] > 2 * perm_df["Desv"]]
print(reales["Variable"].tolist() if len(reales) else "Ninguna: el modelo no depende claramente de ninguna variable.")

**Interpretación.** Una variable es realmente importante si el AUC **cae** de forma clara al desordenarla. Si la caída es cercana a cero o menor que su desviación estándar, el árbol la usa pero no le aporta capacidad predictiva en datos nuevos. Recordar siempre que la importancia indica *asociación en el modelo*, **no causalidad**: que una variable sea importante no demuestra que cause la anemia.

---
# 5. Visualización y comunicación

### 5.1 Matriz de confusión
Se muestra con conteos y en porcentaje por fila (el porcentaje de la fila "Con anemia" es el recall). Se compara el umbral equilibrado (Youden) con el que cumple el recall objetivo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for j, nombre in enumerate(["Youden", f"Recall ≥ {RECALL_OBJETIVO}"]):
    pred = (p_test >= umbrales[nombre]).astype(int)
    for i, norm in enumerate([None, "true"]):
        ConfusionMatrixDisplay.from_predictions(
            y_test, pred, display_labels=["Sin anemia", "Con anemia"],
            normalize=norm, cmap="Blues", ax=axes[i, j],
            values_format=None if norm is None else ".0%", colorbar=False)
        axes[i, j].set_title(f"{nombre} — " + ("conteos" if norm is None else "% por fila real"))
        axes[i, j].grid(False)
plt.tight_layout()
plt.show()

**Interpretación.** Fila superior: cuántos niños caen en cada caso. Fila inferior: el mismo resultado en porcentaje de cada clase real. En la columna de la derecha se ve el costo de exigir más recall: más niños sanos enviados a revisión.

### 5.2 Curva ROC
La curva muestra, para todos los umbrales posibles, la relación entre la tasa de verdaderos positivos (recall) y la de falsos positivos. Se comparan el árbol por defecto, el árbol ajustado y una **regresión logística** como referencia (modelo lineal sencillo, usado como en la versión anterior del cuaderno pero con `scikit-learn`).

In [ ]:
logit = Pipeline([("prep", crear_preprocesador(escalar=True)),
                  ("m", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
p_logit = logit.predict_proba(X_test)[:, 1]
auc_logit = roc_auc_score(y_test, p_logit)

plt.figure(figsize=(7, 6))
for nombre, p in [("Árbol por defecto", p_base),
                  ("Árbol ajustado", p_test),
                  ("Regresión logística (referencia)", p_logit)]:
    f, t, _ = roc_curve(y_test, p)
    plt.plot(f, t, lw=2, label=f"{nombre} (AUC = {roc_auc_score(y_test, p):.3f})")

# Puntos de operación del árbol ajustado
for nombre, c in zip(umbrales, ["black", "green", "crimson"]):
    pred = (p_test >= umbrales[nombre]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    plt.scatter(fp / (fp + tn), tp / (tp + fn), s=70, color=c, zorder=5, label=f"Umbral: {nombre}")

plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC = 0.5)")
plt.xlabel("Tasa de falsos positivos (1 − especificidad)")
plt.ylabel("Tasa de verdaderos positivos (recall)")
plt.title("Curvas ROC en el conjunto de prueba")
plt.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

**Interpretación.** Cuanto más se acerca la curva a la esquina superior izquierda, mejor discrimina el modelo; la diagonal equivale a lanzar una moneda. Los puntos marcados son los tres umbrales evaluados: moverse a lo largo de la curva es justamente el compromiso entre detectar más casos y generar más falsas alarmas. Si el árbol y la regresión logística tienen un AUC parecido, la información de las variables es la limitante, no el tipo de algoritmo.

### 5.3 El árbol entrenado
Se dibujan los primeros niveles (el árbol completo sería ilegible) y debajo se imprimen las reglas en texto. Cada nodo muestra la condición de la división, el *gini/entropy*, el número de niños (*samples*), cuántos hay de cada clase (*value*) y la clase predominante.

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(arbol_final, feature_names=nombres_features,
          class_names=["Sin anemia", "Con anemia"], filled=True, rounded=True,
          max_depth=3, fontsize=9, proportion=False, impurity=False)
plt.title("Árbol de decisión final (primeros 3 niveles)", fontsize=14)
plt.show()

In [ ]:
# Reglas en texto (hasta 4 niveles)
print(export_text(arbol_final, feature_names=list(nombres_features), max_depth=4))

**Interpretación.** Se lee de arriba hacia abajo: en cada nodo, la rama izquierda cumple la condición ("True") y la derecha no. El color indica la clase predominante (más intenso = más pura la hoja). Las primeras divisiones corresponden a las variables que más ayudan a separar los grupos. Un camino completo desde la raíz hasta una hoja es una **regla** del tipo: *"si el niño es de la zona X y no recibe suplemento de hierro, entonces el riesgo de anemia es alto"*, que puede explicarse a personal de salud sin conocimientos de estadística.

### 5.4 Resumen de hallazgos
La siguiente celda genera el resumen con los números reales de esta ejecución.

In [ ]:
mejor = perm_df.iloc[0]
u_r = umbrales[f"Recall ≥ {RECALL_OBJETIVO}"]
m_r = metricas(y_test, p_test, u_r)
m_y = metricas(y_test, p_test, umbrales["Youden"])

parametros_txt = ", ".join(f"{k.replace('arbol__', '')}={v}" for k, v in grid.best_params_.items())
print("RESUMEN DEL MODELO")
print("-" * 70)
print(f"• Datos: {len(df):,} niños; {y.mean():.1%} con algún grado de anemia.")
print(f"• Árbol final: profundidad {arbol_final.get_depth()}, {arbol_final.get_n_leaves()} hojas "
      f"({parametros_txt}).")
print(f"• Discriminación: AUC en prueba = {auc_arbol:.3f} "
      f"(regresión logística de referencia: {auc_logit:.3f}).")
print(f"• Con umbral equilibrado: recall = {m_y['Recall']:.1%}, precision = {m_y['Precision']:.1%}, "
      f"deja sin detectar {int(m_y['Falsos negativos'])} niños con anemia.")
print(f"• Con umbral orientado a recall: detecta {m_r['Recall']:.1%} de los casos, "
      f"a costa de {int(m_r['Falsos positivos'])} falsas alarmas (precision = {m_r['Precision']:.1%}).")
print(f"• Variable con mayor importancia por permutación: {mejor['Variable']} "
      f"(caída de AUC = {mejor['Caída de AUC']:.4f}).")
print()
if auc_arbol < 0.70:
    print("Lectura: el poder predictivo es LIMITADO. Con estas variables el modelo apoya la priorización "
          "pero no sustituye al análisis de hemoglobina; debe usarse como orientación, no como diagnóstico.")
elif auc_arbol < 0.80:
    print("Lectura: el poder predictivo es MODERADO y útil para priorizar niños en riesgo.")
else:
    print("Lectura: el poder predictivo es BUENO con las variables disponibles.")

---
# Conclusiones y limitaciones

**Conclusiones**
- Se construyó un árbol de decisión que clasifica a los niños con/sin anemia, ajustado mediante validación cruzada y evaluado una sola vez sobre datos que nunca vio.
- El árbol ajustado es más simple que el de parámetros por defecto y generaliza mejor; sus reglas pueden comunicarse de forma sencilla.
- La elección del umbral es una **decisión de salud pública**, no solo técnica: depende de cuánto se prioriza no perder casos frente a evitar revisiones innecesarias.

**Limitaciones (conviene mencionarlas en la exposición)**
- Se excluyó la hemoglobina para evitar fuga de información; esto hace el problema más difícil, pero el modelo resultante es honesto sobre lo que se puede predecir *antes* del análisis de sangre.
- La importancia de variables describe **asociación, no causalidad**.
- Los resultados dependen de la calidad y representatividad de los datos; cualquier sesgo en cómo se tomó la muestra se trasladará al modelo.
- Un solo árbol es muy interpretable pero inestable: pequeños cambios en los datos pueden producir un árbol distinto. Como mejora futura se pueden probar *Random Forest* o *Gradient Boosting* y comparar el compromiso entre precisión e interpretabilidad.